# Quantum Acoustic Qubit Stabilization (QuTiP)
Run this notebook in **Google Colab**. It installs QuTiP, runs the Task 3 simulation, and then launches the Task 4 optimizer.

In [0]:
!pip -q install qutip numpy matplotlib scipy

In [0]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
import os, csv, json

out_dir = 'qutip_pipeline_results'
os.makedirs(out_dir, exist_ok=True)

# ---------- Parameters ----------
omega_q = 5.0 * 2*np.pi
omega_a = 2.0 * 2*np.pi
N_a = 10
gamma1 = 1.0e-5 * 2*np.pi
gamma_phi = 1.0e-5 * 2*np.pi
kappa_nat = 1.0e-4 * 2*np.pi

k_B = 1.380649e-23; h = 6.62607015e-34; GHz_to_Hz = 1e9; T_K = 0.015
def n_thermal(freq_GHz, T_K):
    f_hz = freq_GHz * GHz_to_Hz
    x = (h*f_hz)/(k_B*T_K + 1e-300)
    if x>50: return 0.0
    return 1.0/(np.exp(x)-1.0)
n_th = n_thermal(omega_a/(2*np.pi), T_K)

Iq = qt.qeye(2); Ia = qt.qeye(N_a)
sx = qt.sigmax(); sz = qt.sigmaz(); sm = qt.sigmam(); a = qt.destroy(N_a)
sxF = qt.tensor(sx, Ia); szF = qt.tensor(sz, Ia); smF = qt.tensor(sm, Ia)
aF = qt.tensor(Iq, a); aFd = aF.dag()

def build_H(g, eps, delta_a):
    H = 0.5*omega_q*szF + delta_a*(aFd*aF) + 0.5*g*(sxF*(aF+aFd)) + eps*(aF+aFd)
    return H

def cops(kappa_eng):
    k = kappa_nat + kappa_eng
    C = []
    if gamma1>0: C.append(np.sqrt(gamma1)*smF)
    if gamma_phi>0: C.append(np.sqrt(gamma_phi)*szF)
    if k>0:
        C.append(np.sqrt(k*(n_th+1))*aF)
        if n_th>0: C.append(np.sqrt(k*n_th)*aFd)
    return C

def metrics(g, eps, delta_a, kappa_eng):
    H = build_H(g, eps, delta_a)
    C = cops(kappa_eng)
    rho = qt.steadystate(H, C)
    rho_q = rho.ptrace(0)
    pur = float((rho_q*rho_q).tr().real)
    pe = float(rho_q[1,1].real)
    return pur, pe

# ---------- Task 3: coarse sweep ----------
g_list = (np.array([0.5, 1.0, 2.0])*1e-3*2*np.pi)
eps_scale = 1e-4*2*np.pi
eps_list = np.array([0.0, 0.5, 1.0])*eps_scale
delta_list = (np.array([-20, 0, 20])*1e-3*2*np.pi)
kappa_eng_list = (np.array([0.0, 200.0])*1e-6*2*np.pi)
rows = []
best = (-1, None)
for g in g_list:
  for eps in eps_list:
    for da in delta_list:
      for ke in kappa_eng_list:
        pur, pe = metrics(g, eps, da, ke)
        rows.append([g/(2*np.pi)*1e3, eps/eps_scale, da/(2*np.pi)*1e3, ke/(2*np.pi)*1e6, pur, pe])
        if pur>best[0]: best=(pur,(g,eps,da,ke,pe))

with open(os.path.join(out_dir,'sweep.csv'),'w',newline='') as f:
  w=csv.writer(f); w.writerow(['g_MHz','drive_units','delta_MHz','kappa_eng_kHz','purity','P_e'])
  for r in rows: w.writerow([f'{r[0]:.6f}',f'{r[1]:.3f}',f'{r[2]:.6f}',f'{r[3]:.6f}',f'{r[4]:.6f}',f'{r[5]:.6f}'])

print('Best coarse sweep purity:', best[0])
print('Params [g_MHz, drive_units, delta_MHz, kappa_eng_kHz, P_e]:',
      best[1][0]/(2*np.pi)*1e3, best[1][1]/eps_scale, best[1][2]/(2*np.pi)*1e3, best[1][3]/(2*np.pi)*1e6, best[1][4])

# ---------- Task 4: small DE optimization ----------
from scipy.optimize import differential_evolution
toGHz = lambda MHz: MHz*1e-3*2*np.pi
toGHz_kHz = lambda kHz: kHz*1e-6*2*np.pi

def obj(x):
  gM, eu, dM, kk = x
  pur, pe = metrics(toGHz(gM), eu*eps_scale, toGHz(dM), toGHz_kHz(kk))
  return -(pur - 0.1*pe)

bounds=[(0.2,2.0),(0.0,2.0),(-30.0,30.0),(0.0,500.0)]
opt = differential_evolution(obj, bounds, maxiter=12, popsize=10, seed=2, tol=1e-4, polish=True)
gM_opt, eu_opt, dM_opt, kk_opt = opt.x
pur_opt, pe_opt = metrics(toGHz(gM_opt), eu_opt*eps_scale, toGHz(dM_opt), toGHz_kHz(kk_opt))
print('Optimum:', {'g_MHz':gM_opt,'drive_units':eu_opt,'delta_MHz':dM_opt,'kappa_eng_kHz':kk_opt,'purity':pur_opt,'P_e':pe_opt})
with open(os.path.join(out_dir,'opt_summary.json'),'w') as f:
  json.dump({'optimum':{'g_MHz':float(gM_opt),'drive_units':float(eu_opt),'delta_MHz':float(dM_opt),'kappa_eng_kHz':float(kk_opt),'purity':float(pur_opt),'P_e':float(pe_opt)}}, f, indent=2)
